In [1]:
%pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 96.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import mlflow
import polars as pl
import torch
from datasets import Dataset
from mlflow.data.polars_dataset import from_polars
from tqdm.auto import trange
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from utility import load_dotenv

In [3]:
BATCH_SIZE = 128
MAX_LENGTH = 96

translate_model_name = "facebook/nllb-200-distilled-1.3B"
target_lang = "rus_Cyrl"
source_lang = "eng_Latn"
SOURCE_URI = "hf://datasets/ChristophSchuhmann/MS_COCO_2017_URL_TEXT/mscoco.parquet"
HF_REPO_ID = "jstnoname/ms-coco2017-en-ru"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [4]:
load_dotenv()

True

In [5]:
mlflow.set_experiment("ms-coco2017-dataset-en-ru-translate")

<Experiment: artifact_location='mlflow-artifacts:/516920fd4dd54835b0039c5eca1f81ff', creation_time=1792422040491, experiment_id='0', last_update_time=1792422040491, lifecycle_stage='active', name='ms-coco2017-dataset-en-ru-translate', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [6]:
dataset = pl.scan_parquet(SOURCE_URI)
dataset = dataset.with_columns(pl.col("TEXT").alias("en_text"), pl.col("URL").alias("url"))
dataset = dataset.drop("TEXT", "URL")

print("rows:", dataset.select(pl.len()).collect().item())
print("unique en_text:", dataset.select(pl.col("en_text").n_unique()).collect().item())

rows: 591753
unique en_text: 570281


In [7]:
tokenizer = AutoTokenizer.from_pretrained(translate_model_name)
tokenizer.src_lang = source_lang
forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang)

translator = AutoModelForSeq2SeqLM.from_pretrained(translate_model_name).to(device).half()
translator

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.48GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-23): 24 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192, out_features=1024, bias=Tr

In [ ]:
@torch.no_grad()
def translate_texts(texts: list[str]) -> list[str]:
    outputs = []
    for batch_idx in trange(0, len(texts), BATCH_SIZE, desc="Translate"):
        batch = texts[batch_idx : batch_idx + BATCH_SIZE]
        encoded = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=MAX_LENGTH).to(device)
        output_ids = translator.generate(
            **encoded,
            forced_bos_token_id=forced_bos_token_id,
        ).cpu()
        outputs.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    return outputs


unique_texts = dataset.select("en_text").unique().collect().get_column("en_text").sort().to_list()
translated = pl.DataFrame({"en_text": unique_texts, "ru_text": translate_texts(unique_texts)})

In [11]:
dataset = dataset.join(translated.lazy(), on="en_text", how="left").collect()
print("total rows:", dataset.height)

total rows: 591753


In [13]:
hf_dataset = Dataset.from_polars(dataset)
hf_dataset.push_to_hub(HF_REPO_ID, split="all", private=False)
mlflow_dataset = from_polars(dataset)

Setting num_proc from 1 back to 1 for the all split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpc1f6kfym.parquet    :  36%|###6      | 15.9MB / 43.7MB            

Validating                    :           |   0%            

In [14]:
with mlflow.start_run(run_name="translate_dataset") as run:
    mlflow.log_params(
        {
            "batch_size": BATCH_SIZE,
            "model": translate_model_name,
            "source_lang": source_lang,
            "target_lang": target_lang,
            "source_dataset": SOURCE_URI,
            "dataset_repo": HF_REPO_ID,
            "max_length": MAX_LENGTH,
            "dedup": True,
        }
    )
    mlflow.log_metrics(
        {
            "rows": dataset.height,
            "unique_en_text": dataset.select(pl.col("en_text").n_unique()).item(),
            "unique_ru_text": dataset.select(pl.col("ru_text").n_unique()).item(),
            "unique_url": dataset.select(pl.col("url").n_unique()).item(),
        }
    )
    mlflow.log_input(mlflow_dataset, "translating")

🏃 View run translate_dataset at: https://dagshub.com/jstnoname/PhoS.mlflow/#/experiments/0/runs/7c12477510724bfba3869ae9ceea02aa
🧪 View experiment at: https://dagshub.com/jstnoname/PhoS.mlflow/#/experiments/0


In [15]:
dataset.head(5)

en_text,url,ru_text
str,str,str
"""Two adult elephants with a bab…","""http://images.cocodataset.org/…","""Два взрослых слона с слоновым …"
"""A large elephant standing on t…","""http://images.cocodataset.org/…","""Большой слон, стоящий на верши…"
"""an elephant in its pen and som…","""http://images.cocodataset.org/…","""слон в своем загоне и нескольк…"
"""An elephant standing in the di…","""http://images.cocodataset.org/…","""Слон, стоящий в грязи в огорож…"
"""A large elephant stands in its…","""http://images.cocodataset.org/…","""Большой слон стоит в своей оде…"
